# 03 — SVD Matrix Factorization

**Model:** Singular Value Decomposition (SVD)  
**Library:** numpy, scipy  
**Split:** 80% train / 20% test (same split as CF models)

SVD decomposes the user-item rating matrix into latent factors that capture hidden preferences. Instead of comparing users or items directly, it learns a compact representation of both.

---

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from scipy.sparse.linalg import svds

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

PROCESSED_DATA_DIR = '../data/processed/'
RESULTS_DIR        = '../results/'
FIGURES_DIR        = '../results/figures/'

os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(FIGURES_DIR, exist_ok=True)

print('Imports successful.')

---
## 2. Load & Split Data

In [ ]:
ratings = pd.read_csv(os.path.join(PROCESSED_DATA_DIR, 'ratings_clean.csv'))

train_df, test_df = train_test_split(ratings, test_size=0.20, random_state=42)
train_df = train_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print(f'Train size : {len(train_df):,}')
print(f'Test size  : {len(test_df):,}')

---
## 3. Build the User-Item Matrix

In [ ]:
# Pivot training ratings into a user x movie matrix
user_item_matrix = train_df.pivot_table(
    index='userId', columns='movieId', values='rating'
)

# Store means for de-meaning (important for SVD quality)
user_means = user_item_matrix.mean(axis=1)

# Fill NaN with 0 after subtracting user mean (mean-centered matrix)
matrix_demeaned = user_item_matrix.subtract(user_means, axis=0).fillna(0)

print(f'User-item matrix shape: {user_item_matrix.shape}')

---
## 4. Apply SVD

We use `scipy.sparse.linalg.svds` which computes a truncated SVD — only the top K singular values. This is more efficient than full SVD and controls model complexity via the `k` parameter (number of latent factors).

In [ ]:
def run_svd(matrix_demeaned, user_means, n_factors):
    """
    Decompose the mean-centered matrix and reconstruct predicted ratings.
    Returns a DataFrame of predicted ratings (users x movies).
    """
    U, sigma, Vt = svds(matrix_demeaned.values.astype(float), k=n_factors)
    sigma_diag   = np.diag(sigma)

    # Reconstruct and add user means back
    predicted = np.dot(np.dot(U, sigma_diag), Vt)
    predicted += user_means.values.reshape(-1, 1)

    return pd.DataFrame(
        predicted,
        index=matrix_demeaned.index,
        columns=matrix_demeaned.columns
    )

print('SVD function defined.')

---
## 5. Hyperparameter Tuning — Number of Latent Factors

In [ ]:
global_mean = train_df['rating'].mean()
n_factors_list = [5, 10, 20, 50, 100]
tuning_results = []

# Use a validation sample for speed
val_df = test_df.sample(min(1000, len(test_df)), random_state=42)

for n in n_factors_list:
    # Cap n_factors to valid range
    max_factors = min(matrix_demeaned.shape) - 1
    if n >= max_factors:
        print(f'  n_factors={n} exceeds matrix limit ({max_factors}), skipping.')
        continue

    pred_matrix = run_svd(matrix_demeaned, user_means, n)

    preds, actuals = [], []
    for _, row in val_df.iterrows():
        uid, mid = row['userId'], row['movieId']
        if uid in pred_matrix.index and mid in pred_matrix.columns:
            preds.append(pred_matrix.loc[uid, mid])
        else:
            preds.append(global_mean)
        actuals.append(row['rating'])

    rmse = np.sqrt(mean_squared_error(actuals, preds))
    mae  = mean_absolute_error(actuals, preds)
    tuning_results.append({'n_factors': n, 'RMSE': rmse, 'MAE': mae})
    print(f'  n_factors={n:>4}  RMSE: {rmse:.4f}  MAE: {mae:.4f}')

tuning_df = pd.DataFrame(tuning_results)
best_n = tuning_df.loc[tuning_df['RMSE'].idxmin(), 'n_factors']
print(f'\nBest n_factors: {int(best_n)}')

In [ ]:
# Plot RMSE vs number of latent factors
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(tuning_df['n_factors'], tuning_df['RMSE'], marker='o', linewidth=2)
ax.axvline(best_n, color='red', linestyle='--', label=f'Best k={int(best_n)}')
ax.set_xlabel('Number of Latent Factors (k)', fontsize=12)
ax.set_ylabel('RMSE', fontsize=12)
ax.set_title('SVD Tuning: RMSE vs Number of Latent Factors', fontsize=13)
ax.legend()
sns.despine()
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'svd_tuning.png'), dpi=150)
plt.show()
print('Saved svd_tuning.png')

---
## 6. Final SVD Model — Evaluate on Full Test Set

In [ ]:
best_pred_matrix = run_svd(matrix_demeaned, user_means, int(best_n))

preds_svd, actuals_svd = [], []
for _, row in test_df.iterrows():
    uid, mid = row['userId'], row['movieId']
    if uid in best_pred_matrix.index and mid in best_pred_matrix.columns:
        preds_svd.append(best_pred_matrix.loc[uid, mid])
    else:
        preds_svd.append(global_mean)
    actuals_svd.append(row['rating'])

rmse_svd = np.sqrt(mean_squared_error(actuals_svd, preds_svd))
mae_svd  = mean_absolute_error(actuals_svd, preds_svd)

print(f'SVD (k={int(best_n)})  |  RMSE: {rmse_svd:.4f}  |  MAE: {mae_svd:.4f}')

---
## 7. Results Summary

In [ ]:
# Load CF results and append SVD
results_cf  = pd.read_csv(os.path.join(RESULTS_DIR, 'metrics_cf.csv'))
results_svd = pd.DataFrame({
    'Model': [f'SVD (k={int(best_n)})'],
    'RMSE':  [round(rmse_svd, 4)],
    'MAE':   [round(mae_svd, 4)]
})

all_results = pd.concat([results_cf, results_svd], ignore_index=True)
print(all_results.to_string(index=False))

all_results.to_csv(os.path.join(RESULTS_DIR, 'metrics_all.csv'), index=False)
print('\nSaved metrics_all.csv')